## 1.       ABSA

In [2]:
# --- BLOC 1 : extraction d'aspects sur plusieurs avis, sans reseau ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from aspect_sentiment.absa import extract_aspect_candidates

avis_test = [
    "The delivery was slow but the product quality is excellent",
    "Customer service was rude and the refund took forever",
    "Great packaging, arrived in perfect condition, fast shipping",
]

for avis in avis_test:
    aspects = extract_aspect_candidates(avis)
    print(f"Avis : {avis}")
    print(f"  Aspects extraits : {aspects}\n")

Avis : The delivery was slow but the product quality is excellent
  Aspects extraits : ['delivery', 'product quality']

Avis : Customer service was rude and the refund took forever
  Aspects extraits : ['Customer service', 'refund']

Avis : Great packaging, arrived in perfect condition, fast shipping
  Aspects extraits : ['Great packaging', 'condition', 'shipping']



In [ ]:
# --- BLOC 2 : VRAI dataset ABSA (SemEval), remplace les 10 phrases jouets ---
from collections import defaultdict

from aspect_sentiment.absa import load_semeval_absa

(train_textes, train_aspects, train_labels, eval_textes, eval_aspects, eval_labels) = (
    load_semeval_absa()
)

print(f"Train : {len(train_textes)} exemples, Eval : {len(eval_textes)}")
assert (
    len(train_textes) > 0 and len(eval_textes) > 0
), "Aucun exemple charge -- verifie les colonnes affichees ci-dessus."

# verifie la diversite des polarites par aspect (contrairement au jeu jouet)

polarites_par_aspect = defaultdict(set)
for a, ln in zip(train_aspects, train_labels):
    polarites_par_aspect[a].add(ln)
aspects_avec_une_seule_polarite = [
    a for a, pols in polarites_par_aspect.items() if len(pols) == 1
]
print(
    f"Aspects avec une seule polarite vue : "
    f"{len(aspects_avec_une_seule_polarite)} / {len(polarites_par_aspect)}"
)

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/datasets/tomaarsen/setfit-absa-semeval-restaurants/resolve/8885372fe73256f96bb60f65b550538ac5c26047/setfit-absa-semeval-restaurants.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since tomaarsen/setfit-absa-semeval-restaurants couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/ensai/.cache/huggingface/datasets/tomaarsen___setfit-absa-semeval-restaurants/default/0.0.0/8885372fe73256f96bb60f65b550538ac5c26047 (last modified on Wed Aug 19 14:07:22 2026).


Colonnes : ['text', 'span', 'label', 'ordinal']  |  3693 lignes brutes
3602 exemples valides extraits
Train : 2881 exemples, Eval : 721
Aspects avec une seule polarite vue : 878 / 1059


In [4]:
# --- BLOC 3 : entrainement du modele ABSA (classification jointe) ---
from aspect_sentiment.absa import load_absa_classifier, train_absa_model

model, tokenizer = load_absa_classifier(num_labels=3)
print("Parametres du modele ABSA :", sum(p.numel() for p in model.parameters()))

trainer = train_absa_model(
    model,
    tokenizer,
    train_textes,
    train_aspects,
    train_labels,
    eval_textes,
    eval_aspects,
    eval_labels,
    epochs=5,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parametres du modele ABSA : 66955779


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.612256,0.772538,0.659035
2,No log,0.520812,0.793343,0.729534
3,0.551402,0.520426,0.797503,0.726406
4,0.551402,0.548927,0.807212,0.743091
5,0.551402,0.576661,0.804438,0.737955


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
# --- BLOC 4 : evaluation ---
from transformers_arch.fine_tuning import evaluate_fine_tuned_model

resultats = evaluate_fine_tuned_model(trainer)
print("\nResultats d'evaluation ABSA :", resultats)
# A verifier : accuracy et f1 (macro, corrige pour le multi-classe)

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.551402,0.520426,5,0.797503,0.726406



Resultats d'evaluation ABSA : {'eval_loss': 0.5204261541366577, 'eval_accuracy': 0.79750346740638, 'eval_f1': 0.7264059331161762}


In [6]:
# --- BLOC 5 : prediction complete sur un NOUVEL avis multi-aspects ---
from aspect_sentiment.absa import predict_aspect_sentiment

modele_entraine = trainer.model

nouvel_avis = (
    "The delivery was fast this time but customer service still "
    "needs improvement, though the price is fair"
)
aspects_detectes = extract_aspect_candidates(nouvel_avis)
print(f"\nNouvel avis : {nouvel_avis}")
print(f"Aspects detectes automatiquement : {aspects_detectes}")

resultats_finaux = predict_aspect_sentiment(
    nouvel_avis, aspects_detectes, modele_entraine, tokenizer
)
print("\n=== SORTIE FINALE ABSA (aspect -> sentiment) ===")
for aspect, sentiment in resultats_finaux.items():
    print(f"  {aspect:20} -> {sentiment}")

# A retenir : c'est EXACTEMENT la sortie que la problematique du
# projet demande -- pas un sentiment global unique, mais un sentiment
# PAR ASPECT, extrait et classifie automatiquement de bout en bout.


Nouvel avis : The delivery was fast this time but customer service still needs improvement, though the price is fair
Aspects detectes automatiquement : ['delivery', 'time', 'customer service', 'improvement', 'price']

=== SORTIE FINALE ABSA (aspect -> sentiment) ===
  delivery             -> positive
  time                 -> neutral
  customer service     -> negative
  improvement          -> negative
  price                -> positive
